# Cryptography (CC4017) -- Week 4

## Chalenge 1

Consider the following polynomials modulo 2

- x3 + x + 1
- x4 + x + 1
- x4 + x3 + x2 + 1


### 1.1 Start with different initial (non-zero) states and test the periods. What can you conclude about the LFSRs?

- x3 + x + 1: 101 -> 010 -> 100 -> 001 -> 011 -> 111 -> 110 -> 101

- x4 + x + 1: 0001 -> 0011 -> 0111 -> 1111 -> 1110 -> 1101 -> 1010 -> 0101 -> 1011 -> 0110 -> 1100 -> 1001 -> 0010 -> 0100 -> 1000 -> 0001

- x4 + x3 + x2 + 1: 0001 -> 0010 -> 0101 -> 1011 -> 0110 -> 1100 -> 1000 -> 0001

If the polynomial is primitive, then the period of the LSFR will be given by (2^n)-1. If its not primitive, for example, if it is reducible its period will be smaller.

### 1.2 Can you ascertain which is the best polynomial for an LFSR?

The best polynomial for an LFSR would be the second one. General rule, the bigger the state, having more bits, means the better it is for an LFSR. In this case it would be between polynomial number 2 and 3. Since the polynomial 3 is not primitive however, it will have a smaller period, therefore, the second one is the best from the presented ones.

### 1.3 Check if any of these is an irreducible polynomial in sage. What does this say about the polynomial, when used in LFSRs?

The first and the second one are irreducible polynomials,this means that they are a good candidate to be used in a LFSR, because they are probably primitive

## Chalenge 2

Obtain a Python implementation of RC4 from the web and use it to encrypt a file.

mode: encrypt
key: not-so-random-key
imput_file: iam.txt
output_file: output.txt


mode: decrypt
key: not-so-random-key
imput_file: output.txt
output_file: test.txt

In [25]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# author: @manojpandey

import codecs
import sys

MOD = 256

def KSA(key):
    key_length = len(key)
    S = list(range(MOD))  # [0, 1, 2, ... , 255]
    j = 0
    for i in range(MOD):
        j = (j + S[i] + key[i % key_length]) % MOD
        S[i], S[j] = S[j], S[i]  # swap values
    return S

def PRGA(S):
    i = 0
    j = 0
    while True:
        i = (i + 1) % MOD
        j = (j + S[i]) % MOD
        S[i], S[j] = S[j], S[i]  # swap values
        K = S[(S[i] + S[j]) % MOD]
        yield K

def get_keystream(key):
    S = KSA(key)
    return PRGA(S)

def encrypt_logic(key, text):
    # Convert text to a list of ordinals if it's not already in that form
    if isinstance(text, str):
        text = [ord(c) for c in text]  # Convert string to list of unicode values
    elif isinstance(text, bytes):
        text = list(text)  # Convert bytes to a list of integers

    key = [c for c in key]  # key is a list of integers
    keystream = get_keystream(key)

    res = []
    for c in text:
        val = c ^ next(keystream)  # XOR operation
        res.append(val)  # Store the resulting integer
    return bytes(res)  # Return as bytes

def encrypt_file(key, input_file, output_file):
    """ Encrypts the contents of a file and saves it to the output file. """
    with open(input_file, 'rb') as f_in:
        plaintext = f_in.read()
    
    ciphertext = encrypt_logic(key, plaintext)
    
    with open(output_file, 'wb') as f_out:
        f_out.write(ciphertext)  # Write as bytes
        
def decrypt_file(key, input_file, output_file):
    """ Decrypts the contents of a file and saves it to the output file. """
    with open(input_file, 'rb') as f_in:
        ciphertext = f_in.read()
    
    plaintext = encrypt_logic(key, ciphertext)
    
    with open(output_file, 'wb') as f_out:
        f_out.write(plaintext)  # Write as bytes

def main():
    action = input("Choose between encrypt or decrypt: ")
    key = input("Give us the key: ").encode('utf-8')  # Keep the key as bytes
    input_file = input("What file do you want to interact with: ")
    output_file = input("What file do you want to save to: ")

    # Convert the key to a list of integers for KSA
    key = [c for c in key]  # Convert bytes to a list of integers
    
    if action == 'encrypt':
        encrypt_file(key, input_file, output_file)
        print(f"File '{input_file}' encrypted successfully and saved to '{output_file}'")
    elif action == 'decrypt':
        decrypt_file(key, input_file, output_file)
        print(f"File '{input_file}' decrypted successfully and saved to '{output_file}'")
    else:
        print("Invalid action! Use 'encrypt' or 'decrypt'.")
        sys.exit(1)

if __name__ == '__main__':
    main()


File 'output.txt' decrypted successfully and saved to 'test.txt'


## Chalenge 3


Check that this algorithm is compatible with OpenSSL

Firstly converted the key from plain text to hexa, and got the result 6e6f742d736f2d72616e646f6d2d6b6579. After, ran the command with openssl, however since rc4 is deprecated in newer versions the command didnt run. Testing online seemed to present the same results, so yes it is compatible.

## Chalenge 4

Demonstrate with OpenSSL that ChaCha20 produces a repeated ciphertext if you encrypt the same file with the
same key and nonce.

openssl enc -chacha20 -in plaintext.txt -out iam.bin -K 000102030405060708090A0B0C0D0E0F -iv 00000000000000000000000000

openssl enc -chacha20 -in plaintext.txt -out iam2.bin -K 000102030405060708090A0B0C0D0E0F -iv 00000000000000000000000000

cmp iam1.bin iam2.bin

No differences found.


## Challenge 5

In questions 2 and 4, compare the size of the plaintext with the size of the ciphertext. What can you conclude with respect, for example, to AES-CTR and AES-CBC modes studied last week.

In case of the stream ciphers, the size is the same as the plain text, the same happening with AES_CTR. In AES-CBC however, due to the padding addition, the size of the files encrypted files will tend to be bigger, specially using PKCS#7.